# Submit a Spark word-count job to HDFS

The shell cells are complete VM-terminal command blocks. Copy each block into the VM terminal; the in-cluster Jupyter container does not provide the `oc` administration client. Commands use the current OpenShift context and namespace `bigdata`.


## Initialize the VM terminal

Open a terminal in the Linux VM. The one-time functions are installed by `00-Introduction/02-Installation.ipynb`. Load them, connect `oc` to the running CRC cluster, and select the `bigdata` project before continuing.

In [ ]:
source "$HOME/.bashrc"
if ! declare -F okd-env >/dev/null; then
  echo 'Complete 00-Introduction/02-Installation.ipynb first.' >&2
  return 1 2>/dev/null || exit 1
fi
okd-env
oc whoami
oc project
oc get nodes


## 1. Start HDFS and Spark


In [ ]:
%%bash
set -euo pipefail
oc -n bigdata scale statefulset/namenode statefulset/datanode statefulset/spark-master statefulset/spark-worker --replicas=1
for s in namenode datanode spark-master spark-worker; do oc -n bigdata rollout status statefulset/$s --timeout=300s; done


## 2. Prepare the shared HDFS input


In [ ]:
%%bash
set -euo pipefail
oc -n bigdata exec namenode-0 -- sh -c "printf 'spark hadoop spark\nopenshift spark hdfs\nhadoop data\n' > /tmp/spark-words.txt"
oc -n bigdata exec namenode-0 -- hdfs dfs -mkdir -p /training/spark/input
oc -n bigdata exec namenode-0 -- hdfs dfs -put -f /tmp/spark-words.txt /training/spark/input/words.txt
oc -n bigdata exec namenode-0 -- hdfs dfs -rm -r -f /training/spark/spark-word-count
oc -n bigdata exec namenode-0 -- hdfs dfs -cat /training/spark/input/words.txt


## 3. Create the PySpark application in the Jupyter container


In [ ]:
%%bash
set -euo pipefail
cat > .state/spark_hdfs_wordcount.py <<'PY'
from pyspark.sql import SparkSession
from operator import add

spark = SparkSession.builder.appName('hdfs-word-count').getOrCreate()
sc = spark.sparkContext
input_path = 'hdfs://namenode:9000/training/spark/input'
output_path = 'hdfs://namenode:9000/training/spark/spark-word-count'
words = sc.textFile(input_path).flatMap(lambda line: line.split()).map(lambda word: (word.lower(), 1)).reduceByKey(add)
words.sortByKey().map(lambda item: f'{item[0]}\t{item[1]}').saveAsTextFile(output_path)
print('RESULT:', words.sortByKey().collect())
spark.stop()
PY
oc -n bigdata cp .state/spark_hdfs_wordcount.py spark-master-0:/tmp/spark_hdfs_wordcount.py -c jupyter
oc -n bigdata exec spark-master-0 -c jupyter -- sed -n '1,120p' /tmp/spark_hdfs_wordcount.py


## 4. Submit to the standalone Spark master

`--total-executor-cores` limits the total cores used by this application. `--executor-cores` controls cores per executor. The single-worker lab uses one of each.


In [ ]:
%%bash
set -euo pipefail
DRIVER_IP=$(oc -n bigdata get pod spark-master-0 -o jsonpath='{.status.podIP}')
oc -n bigdata exec spark-master-0 -c jupyter -- spark-submit \
  --master spark://spark-master:7077 \
  --deploy-mode client \
  --name hdfs-word-count \
  --total-executor-cores 1 \
  --executor-cores 1 \
  --executor-memory 512m \
  --driver-memory 512m \
  --conf "spark.driver.host=$DRIVER_IP" \
  --conf spark.driver.bindAddress=0.0.0.0 \
  --conf spark.driver.port=7078 \
  --conf spark.blockManager.port=7079 \
  /tmp/spark_hdfs_wordcount.py


## 5. Inspect the HDFS result


In [ ]:
%%bash
set -euo pipefail
oc -n bigdata exec namenode-0 -- hdfs dfs -ls /training/spark/spark-word-count
oc -n bigdata exec namenode-0 -- sh -c "hdfs dfs -cat '/training/spark/spark-word-count/part-*' | sort"


## 6. Inspect Spark application state and logs


In [ ]:
%%bash
set -euo pipefail
oc -n bigdata logs spark-master-0 -c spark-master --tail=50
oc -n bigdata logs spark-worker-0 -c spark-worker --tail=50
echo 'Open the master UI with:'
echo 'oc -n bigdata port-forward --address=127.0.0.1 service/spark-master 18080:8080'
